# Lesson 03 - Agentic Design Patterns

In dieser Lektion erkunden wir drei grundlegende Entwurfsmuster für den Aufbau effektiver KI-Agenten:

1. **Klare Agentenanweisungen** — Präzise, rollen-definierende Prompts verfassen, die das Verhalten des Agenten steuern  
2. **Strukturierte Ausgabe mit Pydantic-Modellen** — Sicherstellen, dass Agenten vorhersehbare, validierte Daten zurückgeben  
3. **Single Responsibility Agents** — Fokusierte Agenten entwerfen, die jeweils eine Sache gut machen

Wir wenden jedes Muster auf ein **Reiseziele-Empfehlungsszenario** an und bauen schrittweise ein System auf, das Ziele vorschlagen, Verfügbarkeiten prüfen und Logistik abwickeln kann.


## Einrichtung


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Muster 1: Klare Agentenanweisungen

Das wirkungsvollste Muster ist auch das einfachste: klare, detaillierte Anweisungen für Ihren Agenten zu schreiben.

Gute Anweisungen definieren:
- **Wer** der Agent ist (Persona und Tonfall)
- **Was** er tun soll (Schritt-für-Schritt-Verantwortlichkeiten)
- **Wie** er sich verhalten soll (Einschränkungen und Stil)

Unten erstellen wir einen ReiseConcierge-Agenten mit expliziten Anweisungen, die jede Antwort prägen, die er erstellt.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Muster 2: Strukturierte Ausgabe mit Pydantic-Modellen

Freiformtext ist für Konversationen nützlich, aber nachgelagerte Systeme benötigen strukturierte Daten.  
Durch die Kombination von **Pydantic-Modellen** mit einer **Werkzeugfunktion** können wir:

- Ein genaues Schema für die Ausgabe des Agents definieren  
- Antworten automatisch validieren  
- Agentenergebnisse zuverlässig in die Anwendungslogik integrieren  

Wir stellen auch ein Werkzeug vor, das Reisezieldetails zurückgibt, sodass der Agent seine Empfehlungen auf reale Daten stützt.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Muster 3: Agenten mit Einzelverantwortung

Komplexe Aufgaben profitieren davon, die Arbeit auf mehrere fokussierte Agenten aufzuteilen, von denen jeder eine einzelne Verantwortung hat:

- Ein **Reiseziel-Experte**, der sich mit Orten und Verfügbarkeiten auskennt
- Ein **Logistikplaner**, der Flüge, Hotels und Reisen organisiert

Dies spiegelt das Prinzip der *Trennung der Verantwortlichkeiten* in der Softwareentwicklung wider — jeder Agent ist dadurch leichter zu testen, zu warten und unabhängig zu verbessern.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Zusammenfassung

In dieser Lektion haben wir drei agentische Entwurfsmuster auf ein Reiseempfehlungsszenario angewendet:

| Muster | Kernidee | Nutzen |
|---|---|---|
| **Klare Anweisungen** | Definieren Sie Persona, Verantwortlichkeiten und Einschränkungen im Voraus | Konsistentes, markenkonformes Agentenverhalten |
| **Strukturierte Ausgabe** | Verwenden Sie Pydantic-Modelle als Antwortformat | Validierte, maschinenlesbare Ergebnisse |
| **Einzelverantwortlichkeit** | Geben Sie jedem Agenten eine fokussierte Aufgabe | Einfacher zu testen, zu warten und zu kombinieren |

Diese Muster lassen sich natürlich zusammenfügen — Sie können klare Anweisungen mit strukturierter Ausgabe in einem Agenten mit Einzelverantwortlichkeit kombinieren, um robuste, produktionsreife Systeme zu erstellen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir auf Genauigkeit achten, kann es bei automatischen Übersetzungen zu Fehlern oder Ungenauigkeiten kommen. Das Originaldokument in seiner ursprünglichen Sprache gilt als maßgebliche Quelle. Für kritische Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Nutzung dieser Übersetzung resultieren.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
